In [22]:
import os
import pandas as pd
import numpy as np

In [23]:
# =============================
# Parameters
# =============================
data_dir = r"D:\hypoglycemia\data"   # Folder containing Normal/ and Tremor/
output_feature_file = r"D:\hypoglycemia\features.csv"
window_size = 200
stride = 100

In [24]:
# =============================
# Feature extraction function
# =============================
def extract_features_from_window(window):
    features = {}
    for col in window.columns:
        features[f'{col}_mean'] = window[col].mean()
        features[f'{col}_std'] = window[col].std()
        features[f'{col}_min'] = window[col].min()
        features[f'{col}_max'] = window[col].max()
        features[f'{col}_energy'] = np.sum(window[col]**2) / len(window)

    # Signal Magnitude Area (SMA)
    if all(axis in window.columns for axis in ['ax','ay','az']):
        sma = np.mean(np.abs(window[['ax','ay','az']])).sum()
        features['sma'] = sma

    return features

In [25]:
# =============================
# Segment and extract features
# =============================
X = []
y = []

for label_folder, label in [('Normal', 0), ('Tremor', 1)]:
    folder_path = os.path.join(data_dir, label_folder)
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)
            df = pd.read_csv(file_path)

            # Drop timestamp column if present
            if 'timestamp' in df.columns:
                df = df.drop(columns=['timestamp'])

            # Convert to numeric
            df = df.apply(pd.to_numeric, errors="coerce")

            # Drop empty columns
            df = df.dropna(axis=1, how="all")

            # Sliding window segmentation
            for start in range(0, len(df) - window_size + 1, stride):
                end = start + window_size
                window = df.iloc[start:end]
                feat = extract_features_from_window(window)
                X.append(feat)
                y.append(label)


In [26]:
# Convert to DataFrame
X = pd.DataFrame(X)
X['label'] = y

In [27]:
X

,ax_mean,ax_std,ax_min,ax_max,ax_energy,ay_mean,ay_std,ay_min,ay_max,ay_energy,...,gy_min,gy_max,gy_energy,gz_mean,gz_std,gz_min,gz_max,gz_energy,sma,label
0,-1619.56,681.717228,-2164,976,3085389.28,-2344.46,946.455980,-4980,-1716,6387792.72,...,-2978,564,85181.580,-210.090,128.332960,-1196,554,60524.810,6279.046667,0
1,-1333.50,722.715308,-2164,976,2297928.08,-2673.82,1073.144428,-4980,-1716,8295194.16,...,-2978,1251,116136.435,-205.815,156.188235,-1196,554,66632.605,6284.620000,0
2,-1501.44,418.789649,-1864,-212,2428829.92,-2244.88,738.362840,-4576,-1732,5581940.00,...,-682,1251,32116.770,-200.115,92.016941,-1030,214,48470.795,6210.173333,0
3,-1437.26,566.985800,-1880,680,2385581.84,-2051.86,379.969082,-3444,-1732,4353784.08,...,-3278,2119,157639.465,-202.070,122.483616,-1247,364,55759.510,6154.226667,0
4,-1408.68,556.299346,-1880,680,2292300.96,-2035.98,390.581887,-3444,-1628,4297006.00,...,-3278,2119,157792.065,-201.165,125.180062,-1247,364,56059.055,6142.446667,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,114.44,254.794205,-592,972,77692.00,-1281.04,1324.593150,-4972,3200,3386837.76,...,-2909,1919,253737.520,-224.600,781.039283,-3559,2951,657417.410,5587.053333,1
232,76.76,399.417946,-1568,1540,164629.12,-1677.22,1786.842551,-4972,2528,5989909.20,...,-2930,3415,467493.780,-209.940,860.980805,-3559,2537,781656.310,5770.766667,1
233,31.32,1027.266470,-5736,4428,1050980.96,-2288.40,1739.212691,-5440,2528,8246511.04,...,-5529,5350,1366336.720,-231.730,1003.077289,-4214,2465,1054832.020,6045.660000,1
234,-64.02,1002.851452,-5736,4428,1004781.04,-1942.14,1780.602117,-5768,2316,6926598.96,...,-5529,5350,1225136.230,-214.220,1046.486547,-4440,3939,1135548.630,5920.753333,1


In [28]:
# Save extracted features
X.to_csv(output_feature_file, index=False)
print(f"Saved {len(X)} segments to: {output_feature_file}")

Saved 236 segments to: D:\hypoglycemia\features.csv


In [31]:
X.columns

Index(['ax_mean', 'ax_std', 'ax_min', 'ax_max', 'ax_energy', 'ay_mean',
       'ay_std', 'ay_min', 'ay_max', 'ay_energy', 'az_mean', 'az_std',
       'az_min', 'az_max', 'az_energy', 'gx_mean', 'gx_std', 'gx_min',
       'gx_max', 'gx_energy', 'gy_mean', 'gy_std', 'gy_min', 'gy_max',
       'gy_energy', 'gz_mean', 'gz_std', 'gz_min', 'gz_max', 'gz_energy',
       'sma', 'label'],
      dtype='object')